In [1]:
!gcloud storage cp -r ./model_wts gs://sam3-app/model_wts

Copying file://./model_wts/config.json to gs://sam3-app/model_wts/config.json
uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composite_upload_enabled False`
If you would like to disable this warning, run:
`gcloud config set storage/parallel_composite_upload_enabled True`
Note that with parallel composite uploads, your object might be
uploaded as a composite object
(https://cloud.google.com/storage/docs/composite-objects), which means
that any user who downloads your object will need to use crc32c
checksums to verify data integrity. gcloud storage is capable of
computing crc32c checksums, but this might pose a problem for other
clients.

Copying file://./model_wts/sam3.pt to gs://sam3-app/model_wts/sam3.pt
Copying file://./model_wts/.ipynb_checkpoints/config-checkpoint.json to gs://sam3-app/model_wts/.ipynb_checkpoints/config-checkpoint.json
  Completed files 34/3 | 3.2GiB/3.2GiB | 238.3MiB/s           

In [3]:
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import sys
import os


# --- Setup Imports ---
# If your notebook is not in the same folder as the 'sam3' package, 
# uncomment the line below and add the path to the parent directory:
# sys.path.append("/path/to/parent/directory")

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# --- Configuration ---
# Update this path to where your model weights are located relative to the notebook
CHECKPOINT_PATH = "./model_wts/sam3.pt" 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Running on device: {DEVICE}")

# --- Load Model (Run once) ---
# We check if 'processor' exists to avoid reloading the model if you re-run the cell
if 'processor' not in locals():
    print("Loading SAM3 Model... (this might take a moment)")
    try:
        model = build_sam3_image_model(
            checkpoint_path=CHECKPOINT_PATH,
            load_from_HF=False,
            device=DEVICE,
        )
        processor = Sam3Processor(model)
        print("SAM3 Model loaded successfully.")
    except Exception as e:
        print(f"Error loading model: {e}")
        # Stop execution if model fails to load
        raise e
else:
    print("Model already loaded.")



/home/jupyter/sam3/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/opt/conda/lib/python3.10/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


Running on device: cpu
Loading SAM3 Model... (this might take a moment)
Error loading model: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx


RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [ ]:
# --- Define Inputs ---
# Change these variables to test different images
IMAGE_PATH = "test_image.jpg"  # Replace with your local image path
PROMPT = "a cat"               # Replace with your text prompt

# --- Inference Logic ---
if not os.path.exists(IMAGE_PATH):
    print(f"Error: Image not found at {IMAGE_PATH}")
else:
    # 1. Load Image
    image = Image.open(IMAGE_PATH).convert("RGB")
    
    # 2. Set Image in Processor
    inference_state = processor.set_image(image)
    
    # 3. Set Text Prompt & Run Inference
    output = processor.set_text_prompt(state=inference_state, prompt=PROMPT)
    
    # 4. Extract Results
    # Output keys: "masks": [N, 1, H, W], "boxes": [N, 4], "scores": [N]
    masks = output["masks"]
    boxes = output["boxes"]
    scores = output["scores"]

    # --- Visualization ---
    # Convert data to CPU/Numpy for plotting
    # We take the best result (index 0) if multiple are returned
    best_mask = masks[0, 0].cpu().numpy() # Shape: (H, W)
    best_box = boxes[0].cpu().numpy()
    best_score = scores[0].item()

    plt.figure(figsize=(12, 6))

    # Plot Original Image
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title(f"Original: {PROMPT}")
    plt.axis("off")

    # Plot Image with Mask Overlay
    plt.subplot(1, 2, 2)
    plt.imshow(image)
    # Overlay mask (using jet colormap with transparency)
    plt.imshow(best_mask, cmap='jet', alpha=0.5)
    
    # Optional: Draw Bounding Box
    x1, y1, x2, y2 = best_box
    rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                         linewidth=2, edgecolor='r', facecolor='none')
    plt.gca().add_patch(rect)
    
    plt.title(f"Prediction (Score: {best_score:.2f})")
    plt.axis("off")

    plt.show()